# Adaptive Multimodal Retrieval & QA Evaluation (Kaggle T4)

Runs the adaptive retrieval pipeline on the benchmark corpus.

**Kaggle Settings**:
- **Accelerator**: GPU T4 (16GB VRAM)
- **Internet**: ON (for pip and model downloading if local flags are enabled)

**Modes of Execution**:
1. **Fast Offline / Cache Mode (Default, ~5-10 seconds)**: Uses precomputed ColVec matrices (`colvec_scores.npy`) and PP-OCRv5 corpus text (`pages_ppocrv5.jsonl`).
2. **Local GPU In-Process Mode**: User explicitly passes `--colvec-local` / `--chandra-local` / `--ppocr-local` (or `--all-local-gpu`) to run live inference directly on the Kaggle T4 GPU.
3. **Remote Endpoint Mode**: Supply `--colvec-endpoint` / `--chandra-endpoint` to query external vLLM microservices.

## 1. Setup Environment, Codebase & Benchmark Dataset

Locates `0.1. BENCHMARK` from Kaggle input datasets and clones/syncs the `adaptive` branch.

In [ ]:
import glob
import os
import shutil
import subprocess

REPO = "https://github.com/iSE-UET-VNU/AXIOM_DE-RD"
BRANCH = "adaptive"
CHECKOUT = "/kaggle/working/repo"

bench = next((p for p in glob.glob("/kaggle/input/*/**/0.1. BENCHMARK", recursive=True) if os.path.isdir(p)), None)
if bench is None:
    bench = next((p for p in glob.glob("/kaggle/input/*/**", recursive=True)
                  if os.path.isdir(p) and os.path.exists(os.path.join(p, "documents.jsonl"))), None)
if bench is None:
    raise SystemExit("Vui lòng gắn dataset chứa 0.1. BENCHMARK (hoặc documents.jsonl) vào notebook.")

bundled = next((p for p in glob.glob("/kaggle/input/*/**/ondemand", recursive=True)
                if os.path.isdir(p) and os.path.exists(os.path.join(p, "adaptive"))), None)

os.makedirs(CHECKOUT, exist_ok=True)
if bundled:
    if not os.path.exists(f"{CHECKOUT}/ondemand"):
        shutil.copytree(bundled, f"{CHECKOUT}/ondemand")
    print("Mã nguồn: Lấy trực tiếp từ dataset bundled ->", bundled)
elif not os.path.exists(f"{CHECKOUT}/ondemand"):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/kaggle/working/clone"], check=True)
    shutil.copytree("/kaggle/working/clone/ondemand", f"{CHECKOUT}/ondemand")
    print("Mã nguồn: Clone từ GitHub branch", BRANCH)

link = os.path.join(CHECKOUT, "0.1. BENCHMARK")
if not os.path.exists(link):
    os.symlink(bench, link)
print("Corpus:", bench, "->", len(os.listdir(os.path.join(link, "data"))), "files")

## 2. Cài đặt Thư viện

Cài đặt các gói tối thiểu cho trích xuất văn bản, layout và adaptive routing.

In [ ]:
!pip install -q pymupdf PyStemmer "pdf-inspector>=0.2.6,<0.3"
# Để chạy in-process Chandra local: uncomment dòng dưới
# !pip install -q "chandra-ocr>=0.2,<0.3"

## 3. Kiểm tra Hệ thống & Tool Health (`doctor`)

Hiển thị thông tin phần cứng Kaggle (GPU T4, VRAM), các cờ in-process khả dụng và trạng thái các bảng điểm ColVec/PP-OCRv5/KDL.

In [ ]:
import sys
sys.path.insert(0, CHECKOUT)
os.chdir(CHECKOUT)

!python -m ondemand.adaptive doctor

## 4. Truy vấn Thử nghiệm Đơn lẻ (Interactive Query Test)

Chạy thử retrieval cho câu hỏi bản vẽ kỹ thuật / bảng biểu phức tạp.

In [ ]:
# Chạy thử với ColVec cache mặc định:
!python -m ondemand.adaptive query "What is the total number of pieces required for the construction of the S-1313 HOG SHELTER?" --ranker page_order

# Chạy thử với cờ GPU in-process (nếu muốn benchmark trực tiếp trên GPU):
# !python -m ondemand.adaptive query "What is the total number of pieces required for the construction of the S-1313 HOG SHELTER?" --colvec-local --ranker page_order

## 5. Chạy Đánh giá Benchmark Toàn diện (Adaptive Evaluation)

Chạy pipeline hoàn chỉnh trên bộ dữ liệu chuẩn (loại trừ 2 câu ground truth lỗi với cờ `--exclude-flawed`).

In [ ]:
!python -m ondemand.adaptive run \
    --tag kaggle_eval_t4 \
    --gate gate_ppocrv5_all.json \
    --pages pages_ppocrv5.jsonl \
    --k fixed:20 \
    --router rule \
    --ranker page_order \
    --exclude-flawed

## 6. Đóng gói & Tải về Kết quả (Download Results)

In [ ]:
import zipfile

WORK_DATA = os.path.join(CHECKOUT, "data/work")
OUT_ZIP = "/kaggle/working/adaptive_results.zip"

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(WORK_DATA):
        if "results" in root:
            for f in files:
                full = os.path.join(root, f)
                zf.write(full, os.path.relpath(full, WORK_DATA))

if os.path.exists(OUT_ZIP):
    print(f"Đã tạo thành công file kết quả: {OUT_ZIP} ({os.path.getsize(OUT_ZIP) / 1024:.1f} KB)")
    print("Bạn có thể tải file này trực tiếp từ tab Output bên phải của giao diện Kaggle.")